# Frente 2 - Transformação dos dados

Briefing 2: Experiência do Cliente. Versão integrada ao `Dataframe_Unido.csv` da nova Frente 1.


**1. Junção dos dados**

O CSV já reúne as tabelas. Nesta etapa separamos avaliações e itens para não somar um item mais de uma vez.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Localiza a tabela criada pela nova Frente 1.
opcoes = [
    Path('Frente1/Dataframe_Unido.csv'),
    Path('../Frente1/Dataframe_Unido.csv'),
    Path('Dataframe_Unido.csv')
]
arquivo = next((p for p in opcoes if p.is_file()), None)
if arquivo is None:
    raise FileNotFoundError(
        'Arquivo Dataframe_Unido.csv não encontrado. Coloque-o na pasta Frente1 '
        'do repositório ou envie para os arquivos da sessão do Colab.'
    )

dados_unidos = pd.read_csv(arquivo, low_memory=False)
print('Arquivo utilizado:', arquivo)
print('Linhas do CSV recebido:', len(dados_unidos))

def montar_base(dados_unidos):
    # [IA] Verificação e conferência - registro 1: adaptação da junção para a nova base da Frente 1.
    dados = dados_unidos.copy()

    # Só pedidos avaliados entram na comparação de notas (1 a 5).
    dados['nota_avaliacao'] = pd.to_numeric(dados['nota_avaliacao'], errors='coerce')
    aval = dados.loc[dados['nota_avaliacao'].between(1, 5)].copy()
    aval['data_criacao_avaliacao'] = pd.to_datetime(aval['data_criacao_avaliacao'], errors='coerce')
    aval['marca_temporal_resposta_avaliacao'] = pd.to_datetime(
        aval['marca_temporal_resposta_avaliacao'], errors='coerce'
    )

    # A mesma avaliação aparece repetida quando há mais de um item no pedido.
    # Contamos avaliações distintas e mantemos a mais recente por pedido.
    quantidade = aval.groupby('id_pedido')['id_avaliacao'].nunique().rename('qtd_avaliacoes')
    aval = (aval.sort_values(['data_criacao_avaliacao', 'marca_temporal_resposta_avaliacao',
                              'id_avaliacao'])
                .drop_duplicates('id_pedido', keep='last'))

    # A junção da Frente 1 pode repetir o mesmo item quando há várias avaliações.
    # Tiramos essa repetição ANTES de somar valores ou contar itens.
    itens = dados.drop_duplicates(['id_pedido', 'id_item_pedido']).copy()

    # Agrupa os itens para calcular a quantidade, os valores e o frete de cada pedido.
    totais = itens.groupby('id_pedido').agg(
        qtd_itens=('id_item_pedido', 'size'),
        qtd_vendedores=('id_vendedor', 'nunique'),
        valor_produtos=('preco', 'sum'),
        valor_frete=('valor_frete', 'sum')
    ).reset_index()
    categoria = (itens.sort_values('preco', ascending=False)
                     .drop_duplicates('id_pedido')[['id_pedido', 'nome_categoria_produto']]
                     .rename(columns={'nome_categoria_produto': 'categoria_produto'}))
    totais = totais.merge(categoria, on='id_pedido', how='left', validate='one_to_one')

    # Informações de pedidos e clientes já vieram juntas no novo CSV.
    colunas_pedido = [
        'id_pedido', 'nota_avaliacao', 'mensagem_comentario_avaliacao',
        'status_pedido', 'marca_temporal_compra_pedido',
        'data_entrega_pedido_transporadora', 'data_entrega_pedido_cliente',
        'data_estimada_entrega_pedido', 'estado_cliente'
    ]
    base = aval[colunas_pedido].merge(totais, on='id_pedido', how='left', validate='one_to_one')
    base = base.merge(quantidade, on='id_pedido', how='left', validate='one_to_one')
    assert base['id_pedido'].is_unique
    return base

base = montar_base(dados_unidos)
print('Pedidos avaliados (únicos):', len(base))
print('Pedidos sem itens:', base['qtd_itens'].isna().sum())
print('Itens únicos no CSV:', len(dados_unidos.drop_duplicates(['id_pedido', 'id_item_pedido'])))


**2. Novos atributos**

Calcula os prazos de entrega, os valores, os comentários e o mês da compra.


In [ ]:
def criar_atributos(base):
    df = base.copy()
    datas = ['marca_temporal_compra_pedido', 'data_entrega_pedido_transporadora',
             'data_entrega_pedido_cliente', 'data_estimada_entrega_pedido']
    for coluna in datas:
        df[coluna] = pd.to_datetime(df[coluna], errors='coerce')

    compra = df['marca_temporal_compra_pedido']
    transportadora = df['data_entrega_pedido_transporadora']
    entrega = df['data_entrega_pedido_cliente']
    previsao = df['data_estimada_entrega_pedido']

    # Datas incoerentes não podem gerar duração negativa. Conserva a falta de data.
    datas_erradas = (transportadora < compra) | (transportadora > entrega)
    df.loc[datas_erradas, 'data_entrega_pedido_transporadora'] = pd.NaT
    transportadora = df['data_entrega_pedido_transporadora']

    # [IA] Verificação e conferência - registro 2: cálculos de tempo de entrega e atraso
    df['tempo_entrega_dias'] = (entrega - compra).dt.total_seconds().div(86400).round(2)
    df['tempo_ate_transportadora_dias'] = (transportadora - compra).dt.total_seconds().div(86400).round(2)
    df['tempo_transporte_dias'] = (entrega - transportadora).dt.total_seconds().div(86400).round(2)
    df['desvio_prazo_dias'] = (entrega.dt.normalize() - previsao.dt.normalize()).dt.days
    df['dias_atraso'] = df['desvio_prazo_dias'].clip(lower=0)
    df['atrasou_entrega'] = (df['desvio_prazo_dias'] > 0).where(df['desvio_prazo_dias'].notna()).astype('Int64')

    df['valor_total_pedido'] = df['valor_produtos'] + df['valor_frete']
    df['proporcao_frete'] = df['valor_frete'] / df['valor_produtos'].replace(0, np.nan)

    mensagem = df['mensagem_comentario_avaliacao'].fillna('').astype(str).str.strip()
    tem_texto = mensagem.ne('') & mensagem.str.lower().ne('vazio')
    df['tem_comentario'] = tem_texto.astype(int)
    df['tamanho_comentario'] = mensagem.where(tem_texto, '').str.len()
    df['nota_baixa'] = df['nota_avaliacao'].isin([1, 2]).astype(int)
    df['grupo_nota'] = np.select(
        [df['nota_avaliacao'].isin([1, 2]), df['nota_avaliacao'].eq(5)],
        ['baixa', 'alta'], default='intermediaria'
    )
    df['situacao_entrega'] = np.select(
        [df['desvio_prazo_dias'].lt(0), df['desvio_prazo_dias'].eq(0), df['desvio_prazo_dias'].gt(0)],
        ['antes_do_prazo', 'no_prazo', 'atrasada'], default='sem_data'
    )
    df['mes_compra'] = compra.dt.to_period('M').astype(str)
    return df

base = criar_atributos(base)
print(base[['nota_avaliacao', 'tempo_entrega_dias', 'dias_atraso', 'tem_comentario']].head())


**3. Codificação, MinMax e Z-score**

Codifica as situações de entrega e aplica MinMax e Z-score.


In [ ]:
def transformar_dados(base):
    df = base.copy()
    # [IA] Verificação e conferência - registro 3: codificação, MinMax e Z-score
    codificadas = pd.get_dummies(df['situacao_entrega'], prefix='entrega', dtype=int)
    df = pd.concat([df, codificadas], axis=1)

    for coluna in ['valor_produtos', 'valor_frete', 'tempo_entrega_dias']:
        valores = df[coluna]
        minimo, maximo = valores.min(), valores.max()
        media, desvio = valores.mean(), valores.std()
        df[coluna + '_minmax'] = (valores - minimo) / (maximo - minimo) if maximo > minimo else 0
        df[coluna + '_zscore'] = (valores - media) / desvio if desvio > 0 else 0
    return df

base = transformar_dados(base)
print(base[['valor_frete_minmax', 'valor_frete_zscore']].head())


**4. Seleção de atributos e arquivos finais**

Seleciona os atributos e confere os resultados finais.


In [ ]:
# Seleciona os atributos usados na análise.
def selecionar_atributos(base):
    colunas_numericas = [
        'qtd_itens', 'qtd_vendedores', 'qtd_avaliacoes', 'valor_produtos', 'valor_frete',
        'valor_total_pedido', 'proporcao_frete', 'tempo_entrega_dias',
        'tempo_ate_transportadora_dias', 'tempo_transporte_dias', 'desvio_prazo_dias',
        'dias_atraso', 'atrasou_entrega', 'tem_comentario', 'tamanho_comentario'
    ]
    colunas_codificadas = [col for col in base.columns if col.startswith('entrega_')]
    colunas_analisadas = colunas_numericas + colunas_codificadas

    # [IA] Verificação e conferência - registro 4: correlação e baixa variância
    correlacoes = base[colunas_analisadas].corrwith(base['nota_avaliacao'])
    variancias = base[colunas_analisadas].var()
    selecao = pd.DataFrame({
        'atributo': colunas_analisadas,
        'correlacao_com_nota': correlacoes.reindex(colunas_analisadas).values,
        'variancia': variancias.reindex(colunas_analisadas).values
    })
    selecao['baixa_variancia'] = selecao['atributo'].isin(colunas_codificadas) & (selecao['variancia'] < 0.01)
    selecao['passa_correlacao'] = selecao['correlacao_com_nota'].abs() >= 0.05

    # As colunas mais importantes para o briefing continuam.
    # As colunas extras passam pelos filtros.
    extras = selecao.loc[
        selecao['atributo'].isin(['qtd_vendedores', 'qtd_avaliacoes']) & selecao['passa_correlacao'],
        'atributo'
    ].tolist()
    extras += selecao.loc[
        selecao['atributo'].isin(colunas_codificadas) & ~selecao['baixa_variancia'],
        'atributo'
    ].tolist()

    colunas_finais = [
        'id_pedido', 'nota_avaliacao', 'grupo_nota', 'nota_baixa', 'mes_compra',
        'estado_cliente', 'categoria_produto', 'status_pedido', 'situacao_entrega',
        'valor_produtos', 'valor_frete', 'valor_total_pedido', 'proporcao_frete',
        'qtd_itens', 'tempo_entrega_dias',
        'tempo_ate_transportadora_dias', 'tempo_transporte_dias', 'desvio_prazo_dias',
        'dias_atraso', 'atrasou_entrega', 'tem_comentario', 'tamanho_comentario',
        'marca_temporal_compra_pedido', 'data_entrega_pedido_cliente', 'data_estimada_entrega_pedido'
    ]
    colunas_escalas = [c + '_' + tipo for c in ['valor_produtos', 'valor_frete', 'tempo_entrega_dias']
                       for tipo in ['minmax', 'zscore']]
    colunas_finais += colunas_escalas + extras
    analitica = base[colunas_finais].copy()

    assert analitica['id_pedido'].is_unique
    assert analitica['nota_avaliacao'].between(1, 5).all()
    assert analitica['atrasou_entrega'].isna().equals(analitica['desvio_prazo_dias'].isna())
    assert analitica['valor_frete_minmax'].dropna().between(0, 1).all()

    return analitica, selecao

analitica, selecao = selecionar_atributos(base)

# Quando o notebook está na raiz, salva em Frente2/. No Colab, salva na pasta atual.
saida = Path('Frente2') if Path('Frente2').is_dir() else Path('.')
analitica.to_csv(saida / 'tabela_analitica_frente2.csv', index=False)
selecao.to_csv(saida / 'selecao_atributos_frente2.csv', index=False)

print('Tabela final:', analitica.shape)
print('Atributos analisados:', len(selecao))
print('Colunas extras:', [c for c in analitica.columns if c.startswith('entrega_') or c in ['qtd_vendedores', 'qtd_avaliacoes']])
print('Baixa variância:', selecao['baixa_variancia'].sum())
print('Saídas salvas em:', saida.resolve())


A Frente 3 deverá executar os tratamentos da nova Frente 1 a partir dos arquivos brutos e depois chamar estas funções da Frente 2. Não deve depender de um CSV tratado pronto.
